In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging as langsmith_logging

# API KEY 정보로드
load_dotenv()
# Langsmith 로깅 설정
langsmith_logging.langsmith("RAG-EXAMPLE-01")


In [ ]:
from langchain.document_loaders import PDFPlumberLoader
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import load_prompt
from langchain_core.runnables import RunnablePassthrough
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

# utils 모듈에서 모든 필요한 기능들을 import
from example.utils import (
    load_pdf_with_toc_filter,
    DebugUpstageAsymmetricEmbeddings,
    get_or_create_vector_store
)

# 문서 파싱 및 목차 필터링
docs = load_pdf_with_toc_filter("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"✅분할된 문서의 수: {len(split_docs)}")

# 임베딩 & 벡터스토어 저장
embeddings = DebugUpstageAsymmetricEmbeddings()
vector_store = get_or_create_vector_store(
    documents=split_docs,
    embedding=embeddings,
)

In [ ]:
# ChatOpenAI 사용
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 프롬프트 로드
prompt = load_prompt("prompts/rag-prompts.yaml")


# 기존 함수도 유지 (호환성을 위해)
def format_docs(docs) -> str:
    """검색된 문서를 프롬프트용 문자열로 직렬화한다."""
    return "\n\n".join(
        f"<document><content>{doc.page_content}</content><page>{doc.metadata.get('page', 'Unknown')}</page><source>{doc.metadata.get('source', 'Unknown')}</source></document>"
        for doc in docs
    )

In [ ]:
# utils에서 import한 모듈들 사용
query_embedder = DebugUpstageAsymmetricEmbeddings()
db = get_or_create_vector_store(
    embedding=query_embedder,
)

retriever = db.as_retriever(search_kwargs={"k": 10})
# compressor = CohereRerank(model="rerank-multilingual-v3.0")
# compression_retriever = ContextualCompressionRetriever(
#     base_compressor=compressor,
#     base_retriever=retriever,
# )
# retriever = db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={"score_threshold": 0.1, "k": 4},  # 이제 거리 기준으로 수정
# )

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
output = chain.invoke("삼성전자에서 개발한 생성형AI에 대해서 설명해줘.")
print(output)

### 전체소스

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging as langsmith_logging

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import load_prompt
from langchain_core.runnables import RunnablePassthrough
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
# utils 모듈에서 모든 필요한 기능들을 import
from example.utils import (
    load_pdf_with_toc_filter,
    DebugUpstageAsymmetricEmbeddings,
    get_or_create_vector_store
)

# API KEY 정보로드
load_dotenv()
# Langsmith 로깅 설정
langsmith_logging.langsmith("RAG-EXAMPLE-02")

# 문서 파싱 및 목차 필터링
docs = load_pdf_with_toc_filter("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"✅분할된 문서의 수: {len(split_docs)}")

# 임베딩 & 벡터스토어 저장
embeddings = DebugUpstageAsymmetricEmbeddings()
vector_store = get_or_create_vector_store(
    documents=split_docs,
    embedding=embeddings,
)

# ChatOpenAI 사용
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 프롬프트 로드
prompt = load_prompt("prompts/rag-prompts.yaml")


# 기존 함수도 유지 (호환성을 위해)
def format_docs(docs) -> str:
    """검색된 문서를 프롬프트용 문자열로 직렬화한다."""
    return "\n\n".join(
        f"<document><content>{doc.page_content}</content><page>{doc.metadata.get('page', 'Unknown')}</page><source>{doc.metadata.get('source', 'Unknown')}</source></document>"
        for doc in docs
    )

# 검색을 위해 임베딩
query_embedder = DebugUpstageAsymmetricEmbeddings()
db = get_or_create_vector_store(
    embedding=query_embedder,
)

LangSmith 추적을 시작합니다.
[프로젝트명]
RAG-EXAMPLE-02


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅파싱된 문서의 수: 23
✅목차 필터링 후 문서의 수: 23
✅내용 감소량: 1895 문자
✅분할된 문서의 수: 41
기존 벡터스토어를 로드했습니다: ./data/chroma
기존 벡터스토어를 로드했습니다: ./data/chroma


In [ ]:
# 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k": 10})
# 리랭커 생성
compressor = CohereRerank(model="rerank-multilingual-v3.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)
retriever = db.as_retriever()
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.2, "k": 4},  # 이제 거리 기준으로 수정
)

# 체인 생성
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)